In [4]:
import yfinance as yf
print(yf.__version__)

1.1.0


In [11]:
import yfinance as yf
import pandas as pd

# Parametry
TICKER = "^GSPC"
START_DATE = "1900-01-01"
END_DATE = None  # None = do dziś
INTERVAL = "1d"  # daily candles

# Pobranie danych
df = yf.download(
    tickers=TICKER,
    start=START_DATE,
    end=END_DATE,
    interval=INTERVAL,
    auto_adjust=False,
    progress=False
)

# 1) Jeśli kolumny są MultiIndex (np. ('Open','^GSPC')), spłaszczamy
if isinstance(df.columns, pd.MultiIndex):
    # najczęściej: pierwszy poziom to nazwa pola (Open/High/...)
    # a drugi to ticker. Bierzemy pierwszy.
    df.columns = [c[0] for c in df.columns]

# Standaryzacja nazw kolumn
df = df.rename(columns={
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Adj Close": "adj_close",
    "Volume": "volume"
})

# Usunięcie wierszy z brakami
df = df.dropna()

print(df.head())
print(df.tail())
print(df.info())


            adj_close      close       high        low       open  volume
Date                                                                     
1927-12-30  17.660000  17.660000  17.660000  17.660000  17.660000       0
1928-01-03  17.760000  17.760000  17.760000  17.760000  17.760000       0
1928-01-04  17.719999  17.719999  17.719999  17.719999  17.719999       0
1928-01-05  17.549999  17.549999  17.549999  17.549999  17.549999       0
1928-01-06  17.660000  17.660000  17.660000  17.660000  17.660000       0
              adj_close        close         high          low         open  \
Date                                                                          
2026-01-26  6950.229980  6950.229980  6964.660156  6921.600098  6923.229980   
2026-01-27  6978.600098  6978.600098  6988.819824  6958.830078  6965.959961   
2026-01-28  6978.029785  6978.029785  7002.279785  6963.459961  7002.000000   
2026-01-29  6969.009766  6969.009766  6992.839844  6870.799805  6977.740234   
2026-01-

In [12]:
# 1. Indeks czasowy
assert isinstance(df.index, pd.DatetimeIndex)
assert df.index.is_monotonic_increasing
assert df.index.is_unique

# 2. Logika OHLC
assert (df["high"] >= df[["open", "close"]].max(axis=1)).all()
assert (df["low"]  <= df[["open", "close"]].min(axis=1)).all()
assert (df["high"] >= df["low"]).all()

# 3. Wolumen
assert (df["volume"] >= 0).all()

print("✅ Data Integrity: podstawowe testy zaliczone")


✅ Data Integrity: podstawowe testy zaliczone


In [13]:
import pandas as pd

def check_session_gaps_daily(df: pd.DataFrame, max_gap_days: int = 4):
    """
    Dla danych dziennych wykrywa podejrzane luki między kolejnymi świecami.
    Domyślnie >4 dni jest podejrzane (bo weekend to 2 dni, długi weekend ~3-4).
    """
    assert isinstance(df.index, pd.DatetimeIndex)

    idx = df.index.sort_values().normalize()
    deltas = idx.to_series().diff().dropna()

    suspicious = deltas[deltas > pd.Timedelta(days=max_gap_days)]
    return {
        "n_rows": len(df),
        "max_gap": deltas.max(),
        "n_suspicious_gaps": len(suspicious),
        "suspicious_gaps": suspicious.head(20),  # pokaże daty i wielkość przerwy
    }

# użycie:
report = check_session_gaps_daily(df, max_gap_days=4)
print(report)


{'n_rows': 24637, 'max_gap': Timedelta('12 days 00:00:00'), 'n_suspicious_gaps': 10, 'suspicious_gaps': Date
1929-12-02    5 days
1933-03-15   12 days
1945-12-26    5 days
1956-12-26    5 days
1958-12-29    5 days
1961-05-31    5 days
1968-07-08    5 days
2001-09-17    7 days
2007-01-03    5 days
2012-10-31    5 days
Name: Date, dtype: timedelta64[ns]}
